In [ ]:
import os
import subprocess
import time
import requests
def job_headers(url):
    return HDR if url.startswith(H + '/') or url == H else {}
def video_duration(path):
    result = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', path],
        check=False, capture_output=True, text=True, timeout=30,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or 'ffprobe failed')
    return float(result.stdout.strip())
while True:
    try:
        r = requests.get(f'{H}/api/worker/next', headers=HDR, timeout=30)
        r.raise_for_status()
        job = r.json().get('job')
        if not job:
            time.sleep(POLL)
            continue
        state['busy'] = True
        state['jobId'] = job['jobId']
        started = time.time()
        input_url = job['inputUrl']
        if not input_url.startswith('http://') and not input_url.startswith('https://'):
            input_url = f'{H}{input_url}'
        in_resp = requests.get(input_url, headers=job_headers(input_url), timeout=120)
        in_resp.raise_for_status()
        with open('/kaggle/working/in.mp4', 'wb') as f:
            f.write(in_resp.content)
        duration = video_duration('/kaggle/working/in.mp4')
        if duration > MAX_DUR + 0.5:
            raise ValueError(f'clip is {duration:.1f}s, limit is {MAX_DUR + 0.5:.1f}s')
        out_dir = '/kaggle/working/out_dir'
        os.makedirs(out_dir, exist_ok=True)
        subprocess.run(
            ['python', 'inference_realesrgan_video.py', '-n', 'realesr-general-x4v3', '-i', '/kaggle/working/in.mp4', '-o', out_dir, '-s', '4', '--suffix', 'out'],
            cwd='/kaggle/working/Real-ESRGAN', check=True, capture_output=True, text=True,
        )
        out_path = '/kaggle/working/out_dir/in_out.mp4'
        if not os.path.exists(out_path):
            raise FileNotFoundError('no output file produced')
        output_url = job['outputUrl']
        if not output_url.startswith('http://') and not output_url.startswith('https://'):
            output_url = f'{H}{output_url}'
        headers = dict(job_headers(output_url))
        headers['Content-Type'] = 'video/mp4'
        with open(out_path, 'rb') as fp:
            payload = fp.read()
        put = requests.put(output_url, data=payload, headers=headers, timeout=300)
        if put.status_code == 403:
            put = requests.put(output_url, data=payload, headers={k: v for k, v in headers.items() if k.lower() != 'content-type'}, timeout=300)
        put.raise_for_status()
        elapsed = time.time() - started
        requests.post(f'{H}/api/worker/{job['"'"'jobId'"'"']}/done', json={'processingSeconds': round(elapsed, 2)}, headers=HDR, timeout=30)
        print(f"[{job['jobId']}] DONE in {elapsed:.1f}s", flush=True)
    except Exception as exc:
        job_id = None
        if 'job' in locals():
            job_id = job.get('jobId')
        if job_id:
            requests.post(f'{H}/api/worker/{job_id}/failed', json={'message': str(exc)[:300]}, headers=HDR, timeout=30)
        print(f"job failed: {exc}", flush=True)
    finally:
        state['busy'] = False
        state['jobId'] = None
        time.sleep(POLL)

In [ ]:
import threading
import time
import requests
last_error = None
def heartbeat_loop():
    global last_error
    while True:
        if not H or not WORKER_SECRET:
            time.sleep(5)
            continue
        try:
            r = requests.post(f'{H}/api/worker/heartbeat', json=state, headers=HDR, timeout=20)
            r.raise_for_status()
            payload = r.json()
            ping_id = payload.get('pingId')
            if ping_id:
                p = requests.post(f'{H}/api/worker/ping/{ping_id}', json={**gpu_info(), **state}, headers=HDR, timeout=20)
                p.raise_for_status()
            msg = None
        except Exception as exc:
            msg = str(exc)
        if msg != last_error:
            if msg:
                print(msg, flush=True)
            last_error = msg
        time.sleep(HB)
threading.Thread(target=heartbeat_loop, daemon=True).start()

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    os.environ['HEROKU_URL'] = client.get_secret('HEROKU_URL').rstrip('/')
    os.environ['WORKER_SECRET'] = client.get_secret('WORKER_SECRET')
except Exception:
    pass
H = os.environ.get('HEROKU_URL', '').rstrip('/')
WORKER_SECRET = os.environ.get('WORKER_SECRET', '')
HDR = {'x-worker-secret': WORKER_SECRET}
POLL = 5
HB = 15
MODEL = 'realesr-general-x4v3'
SCALE = 4
MAX_DUR = 10
state = {'busy': False, 'jobId': None}
def gpu_info():
    try:
        import torch
        if not torch.cuda.is_available():
            return {'available': False, 'name': None, 'vram_mb': None, 'model': MODEL}
        props = torch.cuda.get_device_properties(torch.cuda.current_device())
        return {
            'available': True,
            'name': props.name,
            'vram_mb': int(props.total_memory / (1024 * 1024)),
            'model': MODEL,
        }
    except Exception:
        return {'available': False, 'name': None, 'vram_mb': None, 'model': MODEL}

In [ ]:
%cd /kaggle/working
!rm -rf BasicSR Real-ESRGAN
!git clone https://github.com/XPixelGroup/BasicSR.git
!git clone https://github.com/xinntao/Real-ESRGAN.git
from pathlib import Path
setup_py = Path('/kaggle/working/BasicSR/setup.py')
text = setup_py.read_text()
old = "exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
new = "ns = {}\n        exec(compile(f.read(), version_file, 'exec'), ns)\n    return ns['__version__']"
assert old in text, 'BasicSR setup.py version patch not found'
setup_py.write_text(text.replace(old, new))
!pip install -q --no-deps --no-build-isolation ./BasicSR
!pip install -q addict yapf lmdb ffmpeg-python
!pip install -q realesrgan --no-deps
degradations = Path('/kaggle/working/BasicSR/basicsr/data/degradations.py')
assert degradations.exists(), 'basicsr degradation patch target missing'
degradations.write_text(degradations.read_text().replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional'))
version_file = Path('/kaggle/working/Real-ESRGAN/realesrgan/version.py')
version_file.parent.mkdir(parents=True, exist_ok=True)
version_file.write_text("__version__ = '0.3.0'\n__gitsha__ = 'unknown'\nversion_info = (0, 3, 0)\n")
%cd /kaggle/working/Real-ESRGAN
from realesrgan import RealESRGANer
print('SETUP DONE')